# 价格合适 — 第 7 周练习

## 通过「价格桶平衡采样」改进 QLoRA 微调

**作者：Vagz1216 (Haqs12)**

---

### 要解决的问题

讲师 Day 3/4 训练笔记本直接从 `ed-donner/items_prompts_lite` 取约 20,000 条亚马逊商品的**随机切片**，几乎不做过滤。

但亚马逊目录严重偏向廉价商品：随机 20,000 条里，「$5 手机壳 / $12 线缆」会远多于「$400 电动工具 / $800 电器」。模型在这种不平衡数据上微调后，往往擅长猜便宜货，却因昂贵样本太少而**系统性低估**高价商品。

### 做法：按价格桶均衡采样

作者把第 6 周为 OpenAI 微调写的**价格桶平衡**思路，接到 Week 7 的 QLoRA 开源训练管线：

把训练数据分成 4 个价格桶，并从每桶**等量**抽样：

| 桶 | 价格范围 | 抽样占比 |
|----|----------|----------|
| 预算 | $0 – $50 | 训练集 25% |
| 中档 | $50 – $150 | 训练集 25% |
| 高级 | $150 – $300 | 训练集 25% |
| 豪华 | $300+ | 训练集 25% |

假设：**在更平衡的价格分布上训练，模型在整个价位段都会更准**，而不只是廉价段。

> 在 Google Colab 免费 **T4 GPU** 上运行。  
> 开始前请把 `HF_TOKEN` 与 `WANDB_API_KEY` 配进 Colab Secrets。


---
## 步骤 1：安装库


In [ ]:
# ========== 安装与课程 Day 3/4 对齐的依赖，并拉取评估工具 ==========
# 钉死 bitsandbytes / trl 版本，与讲师训练笔记本一致
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1

# 从课程仓库下载 util.py（含 evaluate / Tester 等）
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

# 安装完成提示
print("Libraries installed!")


In [ ]:
# ========== 导入：数据、训练、量化、可视化与评估 ==========
# os：环境变量（如 WANDB_*）
import os
# re：正则（本格导入保留，后续工具可能用到）
import re
# random：桶内抽样与打乱
import random
# math：数学工具（导入保留）
import math
# defaultdict：按价格桶聚合列表
from collections import defaultdict
# datetime：生成带时间戳的 RUN_NAME
from datetime import datetime
# tqdm：格式化进度条
from tqdm import tqdm
# Colab Secrets
from google.colab import userdata
# Hugging Face Hub 登录
from huggingface_hub import login
# PyTorch
import torch
# transformers 包（导入保留）
import transformers
# 模型 / 分词器 / 4bit 配置 / 设种子
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
# Hugging Face datasets
from datasets import load_dataset, Dataset, DatasetDict
# peft：LoRA 配置与加载适配器
from peft import LoraConfig, PeftModel
# trl：监督微调 Trainer 与配置
from trl import SFTTrainer, SFTConfig
# Weights & Biases 实验跟踪
import wandb
# 画价格分布直方图
import matplotlib.pyplot as plt
# 课程提供的 evaluate()
from util import evaluate

# 打印 PyTorch 版本与 CUDA 是否可用
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
# 打印 GPU 名称（无 CUDA 则 None）
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")


---
## 步骤 2：常量

保持 `LITE_MODE = True`，以适配免费 T4。

与讲师笔记本的主要区别：这里**不直接**用已预格式化的 `items_prompts_lite` 当训练集；  
而是加载原始 `items_lite`，以便读到 `price` 字段，先做价格桶平衡，再自己格式化成 prompt/completion。


In [ ]:
# ========== 常量：模型、数据源、超参、运行名 ==========
# 底座开源模型 id（与讲师相同，勿改）
BASE_MODEL = "meta-llama/Llama-3.2-3B"
# WandB / 项目名前缀
PROJECT_NAME = "price"

# 加载带 price 字段的原始数据集，而不是预格式化 prompts
DATA_USER = "ed-donner"
# 原始商品集：含 price / summary / title 等
RAW_DATASET = f"{DATA_USER}/items_lite"    # Has .price, .summary, .title fields
# 预格式化 prompts：仅用于测试评估，保证和讲师同测试集
PROMPT_DATASET = f"{DATA_USER}/items_prompts_lite"  # Used for test evaluation only

# 轻量模式：卡在 T4 显存预算内
LITE_MODE = True

# 你的 Hugging Face 用户名（推送模型用）
HF_USER = "Haqs12"

# --- 训练 / 模型加载配置 ---
# 默认每次运行动态生成新的训练作业名：
RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}-lite-balanced"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# 审稿提示：若要跳过训练、直接评估作者达到 $79.18 的那次快照，
# 可注释上面 3 行，并取消注释下面 3 行：
# RUN_NAME = "2026-03-11_11.38.39-lite-balanced"
# PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
# HUB_MODEL_NAME = "Haqs12/price-2026-03-11_11.38.39-lite-balanced"

# --- 训练超参（对齐讲师 LITE_MODE）---
EPOCHS = 1
BATCH_SIZE = 32
MAX_SEQUENCE_LENGTH = 128
GRADIENT_ACCUMULATION_STEPS = 1

# --- QLoRA 超参（对齐讲师 LITE_MODE）---
QUANT_4_BIT = True
LORA_R = 32
# 常见约定：alpha = 2 × rank
LORA_ALPHA = LORA_R * 2       # Standard convention: alpha = 2x rank
# 注意力投影层名
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
# 只训注意力层以省 T4 显存
TARGET_MODULES = ATTENTION_LAYERS   # Targeting attention layers only (T4 memory limit)
LORA_DROPOUT = 0.1

# --- 优化器与学习率 ---
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = 'cosine'
OPTIMIZER = "paged_adamw_32bit"

# 探测 GPU 算力代际：Ampere(8.x)+ 可用 bf16
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

# --- 日志与验证规模 ---
VAL_SIZE = 500
LOG_STEPS = 5
SAVE_STEPS = 100
LOG_TO_WANDB = True

# 打印精度选择与将要推送到的 Hub 模型名
print(f"GPU capability: {capability} → {'bfloat16' if use_bf16 else 'float16'}")
print(f"Model will be saved to HuggingFace as: {HUB_MODEL_NAME}")


---
## 步骤 3：登录 Hugging Face 与 Weights & Biases


In [ ]:
# ========== 登录 HF（下数据/推模型）与 WandB（记曲线）==========
# 从 Colab Secrets 取 HF_TOKEN
hf_token = userdata.get('HF_TOKEN')
# 登录 Hub
login(hf_token, add_to_git_credential=True)
print("Logged in to HuggingFace!")

# 取 WANDB_API_KEY 并写入环境变量
wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
# 登录 WandB
wandb.login()
# 项目名、不自动上传模型文件、关闭 watch
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"
print("Logged in to Weights & Biases!")


---
## 步骤 4：加载原始数据并做价格桶平衡抽样

这是本练习的核心改动：不是把随机切片直接丢给 QLoRA，而是先读每条商品的 `price`，分到 4 个桶，再从每桶抽相同数量。

`items_lite` 自带 `price` 字段，便于分桶。讲师用的 `items_prompts_lite` 只有 prompt/completion 字符串，按价格过滤更难。

平衡之后，再用与讲师相同的模板（`What does this cost...` / `Price is $`）格式化成训练对。


In [ ]:
# ========== 加载原始 items_lite 的 train / validation ==========
# 从 Hub 拉取原始商品数据集
raw = load_dataset(RAW_DATASET)
# 训练拆分
raw_train = raw['train']
# 验证拆分
raw_val = raw['validation']

# 打印规模与字段名，确认有 price
print(f"Raw training items: {len(raw_train):,}")
print(f"Raw validation items: {len(raw_val):,}")
print("\nSample item fields:", list(raw_train[0].keys()))


In [ ]:
# ========== 价格分桶 + 每桶等量抽样 ==========
# 与作者第 6 周练习相同的 4 桶体系：强制廉价/昂贵样本均衡出现

def categorize_price(price):
    """把价格归入 4 个桶之一（返回桶标签字符串）。"""
    if price < 50:
        return '$0-50'
    elif price < 150:
        return '$50-150'
    elif price < 300:
        return '$150-300'
    else:
        return '$300+'


def balance_by_price(dataset, items_per_bucket, seed=42):
    """
    按价格桶分组，再从每桶等量抽样。
    返回打乱后的、平衡过的 HuggingFace 行 dict 列表。
    """
    # 固定随机种子，保证可复现
    random.seed(seed)
    # 桶 -> 该桶内样本列表
    buckets = defaultdict(list)

    # 遍历数据集，无效价格跳过
    for item in dataset:
        price = item.get('price')
        # 跳过无价格、非数值或 ≤0 的样本
        if price is None or not isinstance(price, (int, float)) or price <= 0:
            continue
        bucket = categorize_price(price)
        buckets[bucket].append(item)

    # 平衡前各桶数量
    print("Price distribution BEFORE balancing:")
    for b in sorted(buckets.keys()):
        print(f"  {b}: {len(buckets[b]):,} items")

    balanced = []
    # 每桶最多抽 items_per_bucket 条
    for b in sorted(buckets.keys()):
        sample_size = min(items_per_bucket, len(buckets[b]))
        sample = random.sample(buckets[b], sample_size)
        balanced.extend(sample)
        print(f"  Selected {sample_size} items from {b} bucket")

    # 打乱，避免训练时价格成段聚集
    random.shuffle(balanced)  # Shuffle so prices aren't grouped during training
    return balanced


# 训练：每桶 1000 → 共约 4000（仍在 LITE_MODE / T4 预算内）
ITEMS_PER_BUCKET = 1000

print("\n--- Balancing Training Data ---")
balanced_train_items = balance_by_price(raw_train, items_per_bucket=ITEMS_PER_BUCKET)
print(f"\nFinal balanced training set: {len(balanced_train_items):,} items")

print("\n--- Balancing Validation Data ---")
# 验证：每桶 125 → 共约 500
balanced_val_items = balance_by_price(raw_val, items_per_bucket=125)  # 125 per bucket = 500 val items
print(f"Final balanced validation set: {len(balanced_val_items):,} items")


In [ ]:
# ========== 可视化：平衡后训练集的价格直方图 ==========
# 抽出价格列表
train_prices = [item['price'] for item in balanced_train_items]

# 画布
plt.figure(figsize=(12, 4))
# 直方图：应在各价位段大致均匀（桶边界处会有结构）
plt.hist(train_prices, bins=50, color='steelblue', rwidth=0.8)
plt.title(f"Price Distribution of My Balanced Training Set ({len(train_prices):,} items)")
plt.xlabel("Price ($)")
plt.ylabel("Count")
# 三条竖线标出桶边界 50 / 150 / 300
plt.axvline(x=50, color='red', linestyle='--', alpha=0.7, label='Bucket boundaries')
plt.axvline(x=150, color='red', linestyle='--', alpha=0.7)
plt.axvline(x=300, color='red', linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()

# 打印 min / max / mean
print(f"Price stats: min=${min(train_prices):.2f}, max=${max(train_prices):.2f}, "
      f"mean=${sum(train_prices)/len(train_prices):.2f}")


---
## 步骤 5：格式化为 prompt / completion 对

把平衡后的商品 dict，转成讲师 `SFTTrainer` 期望的 `prompt` / `completion` 字符串。

模板与 `items.py` 一致，因此模型看到的文本结构不变，只是训练切片更均衡。


In [ ]:
# ========== 与 week7/pricer/items.py 对齐的问答模板 ==========
# 提问句（行为相关字符串，保持英文原文）
QUESTION = "What does this cost to the nearest dollar?"
# 价格前缀（模型续写数字的锚点）
PREFIX = "Price is $"


def make_prompt_completion(item, tokenizer, max_tokens=128, include_answer=True):
    """
    把原始商品 dict 转成 SFTTrainer 需要的 prompt/completion。
    逻辑对齐讲师 items.py 的 make_prompts()。
    """
    # 优先 summary，否则 title
    summary = item.get('summary') or item.get('title', '')

    # 按 token 数截断摘要（类似讲师 CUTOFF）
    tokens = tokenizer.encode(summary, add_special_tokens=False)
    if len(tokens) > max_tokens:
        summary = tokenizer.decode(tokens[:max_tokens]).rstrip()

    # 拼完整 prompt：问题 + 摘要 + Price is $
    prompt = f"{QUESTION}\n\n{summary}\n\n{PREFIX}"

    if include_answer:
        # 训练/验证：completion 是「整数美元.00」
        completion = f"{round(item['price'])}.00"
    else:
        # 测试路径：保留原始 price 字符串（本练习主要用 include_answer=True）
        completion = str(item['price'])

    return {"prompt": prompt, "completion": completion}


# 先加载分词器，格式化时要用它截断
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Formatting balanced training data...")
train_formatted = [make_prompt_completion(item, tokenizer, include_answer=True)
                   for item in tqdm(balanced_train_items)]

print("Formatting balanced validation data...")
val_formatted = [make_prompt_completion(item, tokenizer, include_answer=True)
                 for item in tqdm(balanced_val_items)]

# 转成 HuggingFace Dataset，供 SFTTrainer 使用
train_dataset = Dataset.from_list(train_formatted)
val_dataset = Dataset.from_list(val_formatted)

print(f"\nTraining dataset: {len(train_dataset):,} items")
print(f"Validation dataset: {len(val_dataset):,} items")
print("\nSample formatted training item:")
print(train_dataset[0]['prompt'])
print("Completion:", train_dataset[0]['completion'])


---
## 步骤 6：以 4bit 量化加载底座模型

与讲师 Day 3/4 相同：用 **4bit NF4** 加载 Llama 3.2 3B，塞进 T4 约 16GB 显存预算。


In [ ]:
# ========== 4bit 量化配置并加载冻结底座 ==========
# 与讲师设置一致的 BitsAndBytes 4bit 配置
if QUANT_4_BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        # 按 GPU 能力在 bf16 / fp16 间切换
        bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
        bnb_4bit_quant_type="nf4"
    )

# 加载底座因果 LM（权重冻结，后续靠 LoRA 适配）
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
# 生成时用分词器的 pad_token_id
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Base model loaded! Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")


---
## 步骤 7：配置 QLoRA 并训练

LoRA / SFT 超参与讲师 `LITE_MODE` 对齐。  
**唯一自变量**是训练数据：这里的价格平衡集，而不是随机切片。


In [ ]:
# ========== LoRA 适配器配置（对齐讲师 LITE_MODE）==========
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,    # Scaling factor for the adapter updates
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,                 # Rank of the low-rank decomposition (32 for LITE_MODE)
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,  # Only attention layers to save T4 memory
)

# 打印 rank / alpha / 目标模块
print(f"LoRA config: r={LORA_R}, alpha={LORA_ALPHA}, targets={TARGET_MODULES}")


In [ ]:
# ========== WandB init + SFTConfig（对齐讲师 LITE_MODE）==========
# 若开启日志，则初始化一次 WandB run
if LOG_TO_WANDB:
    wandb.init(project=PROJECT_NAME, name=RUN_NAME)

# 监督微调训练参数：batch、优化器、精度、Hub 推送、评估步数等
train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,       # Auto-saves to HuggingFace every SAVE_STEPS — crash-safe!
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    dataset_text_field="prompt",
)

print(f"Training config ready. Will push checkpoints to: {HUB_MODEL_NAME}")


In [ ]:
# ========== 组装 SFTTrainer、开训、推送、结束 WandB ==========
# 把底座模型、平衡后的 train/val、LoRA 配置与训练参数接起来
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train_dataset,      # My price-balanced training data
    eval_dataset=val_dataset,          # My price-balanced validation data
    peft_config=lora_parameters,
    args=train_parameters
)

# 启动训练（免费 T4 + 约 4000 条大约 10–20 分钟）
fine_tuning.train()

# 训练结束再推一次最终适配器到 Hub（private）
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"\nTraining complete! Model saved to: {HUB_MODEL_NAME}")

# 关闭 WandB run
if LOG_TO_WANDB:
    wandb.finish()


---
## 步骤 8：评估 — 平衡训练数据有没有帮助？

用与讲师相同的约 200 条测试集评估价格平衡微调模型，便于直接对比基线分数。


In [ ]:
# ========== 加载标准测试拆分（与讲师同集，保证可比）==========
prompt_ds = load_dataset(PROMPT_DATASET)
test = prompt_ds['test']
print(f"Test items: {len(test):,}")


In [ ]:
# ========== 从 Hub 加载刚训好的 Peft 适配器到底座上 ==========
# 若中途崩溃后只跑评估，请把 HUB_MODEL_NAME 改成你的仓库
fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)
print(f"Fine-tuned model loaded! Memory: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")


In [ ]:
# ========== 推理函数：与讲师 Day 5 同款贪婪解码 ==========
def model_predict(item):
    """
    标准推理：对齐讲师 Day 5 的 model_predict()。
    用相同贪婪解码，使分数差异尽量只来自训练数据是否平衡。
    """
    # 对测试 prompt 分词并放到 CUDA
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    with torch.no_grad():
        # 半精度 autocast 加速生成
        with torch.autocast("cuda", dtype=torch.float16):
            output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    # 只取新生成段
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)


# 先对 3 条做健全性检查
print("Sanity check:")
for i in range(3):
    pred = model_predict(test[i]).strip()
    actual = test[i]['completion']
    print(f"  Item {i+1}: Predicted '{pred}' | Actual: ${actual}")


In [ ]:
# ========== 全量评估：200 条未见测试样本 ==========
print("Running full evaluation (200 test items)...")
# 固定种子，便于复现评估过程中的随机性
set_seed(42)
# 课程 util.evaluate：汇总误差等指标
evaluate(model_predict, test)


---
## 结果与反思

| 模型 | 训练数据 | 平均误差 ($) |
|------|----------|--------------|
| 讲师基线（随机样本） | `items_prompts_lite`（20,000 条） | 65.40 |
| 本模型（价格平衡样本） | `items_lite` 按桶平衡（4,000 条） | 79.18 |

### 观察

在仅 4,000 条价格平衡数据上微调后，平均误差为 **$79.18**，逊于讲师精简微调基线（$65.40）。但有重要前提：讲师模型用了 **20,000** 条——大约 5 倍数据量。

即便如此，本模型仍明显好于人类基线（$87.62）以及 Base Llama 3.2 4bit（$110.72）（见课程综合评估表）。

核心收获：**数据质量很重要**。亚马逊随机样本以廉价商品为主；强制每个价格层样本量接近，模型会看到更具代表性的价位覆盖。

后续可把同一平衡策略扩到 20,000 条（可能需要付费 GPU 扛住 batch 显存），预期在高价段精度上更能追上甚至超过随机样本基线。

这直接呼应第 6 周数据工程课的观点：

> “数据管理可以被认为是数据科学家的一项不太光彩的工作。  
> 我说那是废话！这就是科学发生的地方。” — Ed Donner

---
_第 7 周顶点练习 | Vagz1216 | Andela AI Engineering Bootcamp_
